In [0]:
dbutils.widgets.text("catalog", "tpch_dev", "Catalog Name")
catalog = dbutils.widgets.get("catalog")

In [0]:
# silver.region
from pyspark.sql.functions import *
from pyspark.sql.window import Window

region_df = spark.read.table(f"{catalog}.bronze.region")
standardized_region_df = (
    region_df
    .drop("_source_file", "_ingested_at", "_pipeline_run_id")
    .withColumns({
        "r_name": trim(col("r_name")),
        "r_comment": trim(col("r_comment"))
    })
)

# Flag records for quarantine: r_regionkey is null or duplicate
w = Window.partitionBy("r_regionkey")
standardized_region_df = standardized_region_df.withColumn(
    "_key_count", count("*").over(w)
)

standardized_region_df = standardized_region_df.withColumn(
    "_quarantine_reason",
    nullif(concat_ws(
        ", ",
        when(col("r_regionkey").isNull(), lit("r_regionkey is null")).otherwise(lit(None).cast("string")),
        when(col("_key_count") > 1, lit("duplicate r_regionkey")).otherwise(lit(None).cast("string"))
    ), lit(""))
)

quarantine_df = standardized_region_df.filter(
    col("_quarantine_reason").isNotNull()
).drop("_key_count")

clean_df = standardized_region_df.filter(
    col("r_regionkey").isNotNull() & (col("_key_count") == 1)
).drop("_key_count", "_quarantine_reason")

# Write quarantined records
quarantine_df.write.mode("overwrite").saveAsTable(f"{catalog}.silver.region_quarantine")

# Write clean records
clean_df.write.mode("overwrite").saveAsTable(f"{catalog}.silver.region")

In [0]:
# silver.nation
from pyspark.sql.functions import *
from pyspark.sql.window import Window

nation_df = spark.read.table(f"{catalog}.bronze.nation")
standardized_nation_df = (
    nation_df
    .drop("_source_file", "_ingested_at", "_pipeline_run_id")
    .withColumns({
        "n_name": trim(col("n_name")),
        "n_comment": trim(col("n_comment"))
    })
)

# Flag records for quarantine: n_nationkey is null or duplicate
w = Window.partitionBy("n_nationkey")
standardized_nation_df = standardized_nation_df.withColumn(
    "_key_count", count("*").over(w)
)

# Referential integrity: n_regionkey must match r_regionkey in silver.region
valid_region_df = spark.read.table(f"{catalog}.silver.region").select(
    col("r_regionkey").alias("_valid_regionkey")
)
standardized_nation_df = standardized_nation_df.join(
    valid_region_df,
    standardized_nation_df.n_regionkey == valid_region_df._valid_regionkey,
    "left"
).withColumn(
    "_invalid_regionkey",
    when(
        col("n_regionkey").isNotNull() & col("_valid_regionkey").isNull(),
        lit("n_regionkey does not match any r_regionkey in silver.region")
    ).otherwise(lit(None).cast("string"))
)

standardized_nation_df = standardized_nation_df.withColumn(
    "_quarantine_reason",
    nullif(concat_ws(
        ", ",
        when(col("n_nationkey").isNull(), lit("n_nationkey is null")).otherwise(lit(None).cast("string")),
        when(col("_key_count") > 1, lit("duplicate n_nationkey")).otherwise(lit(None).cast("string")),
        when(col("_invalid_regionkey").isNotNull(), col("_invalid_regionkey")).otherwise(lit(None).cast("string"))
    ), lit(""))
)

quarantine_nation_df = standardized_nation_df.filter(
    col("_quarantine_reason").isNotNull()
).drop("_key_count", "_valid_regionkey", "_invalid_regionkey")

clean_nation_df = standardized_nation_df.filter(
    col("n_nationkey").isNotNull() & (col("_key_count") == 1) & col("_valid_regionkey").isNotNull()
).drop("_key_count", "_quarantine_reason", "_valid_regionkey", "_invalid_regionkey")

# Write quarantined records
quarantine_nation_df.write.mode("overwrite").saveAsTable(f"{catalog}.silver.nation_quarantine")

# Write clean records
clean_nation_df.write.mode("overwrite").saveAsTable(f"{catalog}.silver.nation")

In [0]:
# silver.part
from pyspark.sql.functions import *
from pyspark.sql.window import Window

part_df = spark.read.table(f"{catalog}.bronze.part")
standardized_part_df = (
    part_df
    .drop("_source_file", "_ingested_at", "_pipeline_run_id")
    .withColumns({
        "p_name": trim(col("p_name")),
        "p_mfgr": trim(col("p_mfgr")),
        "p_brand": trim(col("p_brand")),
        "p_type": trim(col("p_type")),
        "p_container": trim(col("p_container")),
        "p_comment": trim(col("p_comment"))
    })
)

# Flag records for quarantine: p_partkey is null or duplicate
w = Window.partitionBy("p_partkey").orderBy("p_partkey")
standardized_part_df = standardized_part_df.withColumn(
    "_key_count", count("*").over(w)
).withColumn(
    "_row_num", row_number().over(w)
)

standardized_part_df = standardized_part_df.withColumn(
    "_quarantine_reason",
    nullif(concat_ws(
        ", ",
        when(col("p_partkey").isNull(), lit("p_partkey is null")).otherwise(lit(None).cast("string")),
        when((col("_key_count") > 1) & (col("_row_num") > 1), lit("duplicate p_partkey")).otherwise(lit(None).cast("string"))
    ), lit(""))
)

quarantine_df = standardized_part_df.filter(
    col("_quarantine_reason").isNotNull()
).drop("_key_count", "_row_num")

clean_df = standardized_part_df.filter(
    col("p_partkey").isNotNull() & (col("_row_num") == 1)
).drop("_key_count", "_quarantine_reason", "_row_num")

# Write quarantined records
quarantine_df.write.mode("overwrite").saveAsTable(f"{catalog}.silver.part_quarantine")

# Write clean records
clean_df.write.mode("overwrite").saveAsTable(f"{catalog}.silver.part")

In [0]:
# silver.supplier
from pyspark.sql.functions import *
from pyspark.sql.window import Window

supplier_df = spark.read.table(f"{catalog}.bronze.supplier")
standardized_supplier_df = (
    supplier_df
    .drop("_source_file", "_ingested_at", "_pipeline_run_id")
    .withColumns({
        "s_name": trim(col("s_name")),
        "s_address": trim(col("s_address")),
        "s_phone": trim(col("s_phone")),
        "s_comment": trim(col("s_comment"))
    })
)

# Flag records for quarantine: s_suppkey is null or duplicate
w = Window.partitionBy("s_suppkey").orderBy("s_suppkey")
standardized_supplier_df = standardized_supplier_df.withColumn(
    "_key_count", count("*").over(w)
).withColumn(
    "_row_num", row_number().over(w)
)

# Referential integrity: s_nationkey must match n_nationkey in silver.nation
valid_nation_df = spark.read.table(f"{catalog}.silver.nation").select(
    col("n_nationkey").alias("_valid_nationkey")
)
standardized_supplier_df = standardized_supplier_df.join(
    valid_nation_df,
    standardized_supplier_df.s_nationkey == valid_nation_df._valid_nationkey,
    "left"
).withColumn(
    "_invalid_nationkey",
    when(
        col("s_nationkey").isNotNull() & col("_valid_nationkey").isNull(),
        lit("s_nationkey does not match any n_nationkey in silver.nation")
    ).otherwise(lit(None).cast("string"))
)

standardized_supplier_df = standardized_supplier_df.withColumn(
    "_quarantine_reason",
    nullif(concat_ws(
        ", ",
        when(col("s_suppkey").isNull(), lit("s_suppkey is null")).otherwise(lit(None).cast("string")),
        when((col("_key_count") > 1) & (col("_row_num") > 1), lit("duplicate s_suppkey")).otherwise(lit(None).cast("string")),
        when(col("_invalid_nationkey").isNotNull(), col("_invalid_nationkey")).otherwise(lit(None).cast("string"))
    ), lit(""))
)

quarantine_supplier_df = standardized_supplier_df.filter(
    col("_quarantine_reason").isNotNull()
).drop("_key_count", "_valid_nationkey", "_invalid_nationkey", "_row_num")

clean_supplier_df = standardized_supplier_df.filter(
    col("s_suppkey").isNotNull() & (col("_row_num") == 1) & col("_valid_nationkey").isNotNull()
).drop("_key_count", "_quarantine_reason", "_valid_nationkey", "_invalid_nationkey", "_row_num")

# Write quarantined records
quarantine_supplier_df.write.mode("overwrite").saveAsTable(f"{catalog}.silver.supplier_quarantine")

# Write clean records
clean_supplier_df.write.mode("overwrite").saveAsTable(f"{catalog}.silver.supplier")

In [0]:
# silver.partsupp
from pyspark.sql.functions import *
from pyspark.sql.window import Window

partsupp_df = spark.read.table(f"{catalog}.bronze.partsupp")
standardized_partsupp_df = (
    partsupp_df
    .drop("_source_file", "_ingested_at", "_pipeline_run_id")
    .withColumns({
        "ps_comment": trim(col("ps_comment"))
    })
)

# Flag records for quarantine: ps_partkey or ps_suppkey is null, or duplicate composite key
w = Window.partitionBy("ps_partkey", "ps_suppkey").orderBy("ps_partkey", "ps_suppkey")
standardized_partsupp_df = standardized_partsupp_df.withColumn(
    "_key_count", count("*").over(w)
).withColumn(
    "_row_num", row_number().over(w)
)

# Referential integrity: ps_partkey must match p_partkey in silver.part
valid_part_df = spark.read.table(f"{catalog}.silver.part").select(
    col("p_partkey").alias("_valid_partkey")
)
# Referential integrity: ps_suppkey must match s_suppkey in silver.supplier
valid_supplier_df = spark.read.table(f"{catalog}.silver.supplier").select(
    col("s_suppkey").alias("_valid_suppkey")
)

standardized_partsupp_df = standardized_partsupp_df.join(
    valid_part_df,
    standardized_partsupp_df.ps_partkey == valid_part_df._valid_partkey,
    "left"
).join(
    valid_supplier_df,
    standardized_partsupp_df.ps_suppkey == valid_supplier_df._valid_suppkey,
    "left"
).withColumn(
    "_invalid_partkey",
    when(
        col("ps_partkey").isNotNull() & col("_valid_partkey").isNull(),
        lit("ps_partkey does not match any p_partkey in silver.part")
    ).otherwise(lit(None).cast("string"))
).withColumn(
    "_invalid_suppkey",
    when(
        col("ps_suppkey").isNotNull() & col("_valid_suppkey").isNull(),
        lit("ps_suppkey does not match any s_suppkey in silver.supplier")
    ).otherwise(lit(None).cast("string"))
)

standardized_partsupp_df = standardized_partsupp_df.withColumn(
    "_quarantine_reason",
    nullif(concat_ws(
        ", ",
        when(col("ps_partkey").isNull(), lit("ps_partkey is null")).otherwise(lit(None).cast("string")),
        when(col("ps_suppkey").isNull(), lit("ps_suppkey is null")).otherwise(lit(None).cast("string")),
        when((col("_key_count") > 1) & (col("_row_num") > 1), lit("duplicate ps_partkey/ps_suppkey")).otherwise(lit(None).cast("string")),
        when(col("_invalid_partkey").isNotNull(), col("_invalid_partkey")).otherwise(lit(None).cast("string")),
        when(col("_invalid_suppkey").isNotNull(), col("_invalid_suppkey")).otherwise(lit(None).cast("string"))
    ), lit(""))
)

quarantine_partsupp_df = standardized_partsupp_df.filter(
    col("_quarantine_reason").isNotNull()
).drop("_key_count", "_row_num", "_valid_partkey", "_valid_suppkey", "_invalid_partkey", "_invalid_suppkey")

clean_partsupp_df = standardized_partsupp_df.filter(
    col("ps_partkey").isNotNull()
    & col("ps_suppkey").isNotNull()
    & (col("_row_num") == 1)
    & col("_valid_partkey").isNotNull()
    & col("_valid_suppkey").isNotNull()
).drop("_key_count", "_quarantine_reason", "_row_num", "_valid_partkey", "_valid_suppkey", "_invalid_partkey", "_invalid_suppkey")

# Write quarantined records
quarantine_partsupp_df.write.mode("overwrite").saveAsTable(f"{catalog}.silver.partsupp_quarantine")

# Write clean records
clean_partsupp_df.write.mode("overwrite").saveAsTable(f"{catalog}.silver.partsupp")

In [0]:
# silver.customers
from pyspark.sql.functions import *
from pyspark.sql.window import Window

customers_df = spark.read.table(f"{catalog}.bronze.customers")
standardized_customers_df = (
    customers_df
    .drop("_source_file", "_ingested_at", "_pipeline_run_id")
    .withColumns({
        "c_name": trim(col("c_name")),
        "c_address": trim(col("c_address")),
        "c_phone": trim(col("c_phone")),
        "c_mktsegment": trim(col("c_mktsegment")),
        "c_comment": trim(col("c_comment"))
    })
)

# Flag records for quarantine: c_custkey is null or duplicate
w = Window.partitionBy("c_custkey").orderBy("c_custkey")
standardized_customers_df = standardized_customers_df.withColumn(
    "_key_count", count("*").over(w)
).withColumn(
    "_row_num", row_number().over(w)
)

# Referential integrity: c_nationkey must match n_nationkey in silver.nation
valid_nation_df = spark.read.table(f"{catalog}.silver.nation").select(
    col("n_nationkey").alias("_valid_nationkey")
)
standardized_customers_df = standardized_customers_df.join(
    valid_nation_df,
    standardized_customers_df.c_nationkey == valid_nation_df._valid_nationkey,
    "left"
).withColumn(
    "_invalid_nationkey",
    when(
        col("c_nationkey").isNotNull() & col("_valid_nationkey").isNull(),
        lit("c_nationkey does not match any n_nationkey in silver.nation")
    ).otherwise(lit(None).cast("string"))
)

standardized_customers_df = standardized_customers_df.withColumn(
    "_quarantine_reason",
    nullif(concat_ws(
        ", ",
        when(col("c_custkey").isNull(), lit("c_custkey is null")).otherwise(lit(None).cast("string")),
        when((col("_key_count") > 1) & (col("_row_num") > 1), lit("duplicate c_custkey")).otherwise(lit(None).cast("string")),
        when(col("_invalid_nationkey").isNotNull(), col("_invalid_nationkey")).otherwise(lit(None).cast("string"))
    ), lit(""))
)

quarantine_customers_df = standardized_customers_df.filter(
    col("_quarantine_reason").isNotNull()
).drop("_key_count", "_valid_nationkey", "_invalid_nationkey", "_row_num")

clean_customers_df = standardized_customers_df.filter(
    col("c_custkey").isNotNull() & (col("_row_num") == 1) & col("_valid_nationkey").isNotNull()
).drop("_key_count", "_quarantine_reason", "_valid_nationkey", "_invalid_nationkey", "_row_num")

# Write quarantined records
quarantine_customers_df.write.mode("overwrite").saveAsTable(f"{catalog}.silver.customers_quarantine")

# Write clean records
clean_customers_df.write.mode("overwrite").saveAsTable(f"{catalog}.silver.customers")

In [0]:
# silver.orders
from pyspark.sql.functions import *
from pyspark.sql.window import Window

orders_df = spark.read.table(f"{catalog}.bronze.orders")
standardized_orders_df = (
    orders_df
    .drop("_source_file", "_ingested_at", "_pipeline_run_id")
    .withColumns({
        "o_orderstatus": trim(col("o_orderstatus")),
        "o_orderdate": try_to_date(col("o_orderdate"), "yyyy-MM-dd"),
        "o_orderpriority": trim(col("o_orderpriority")),
        "o_clerk": trim(col("o_clerk")),
        "o_comment": trim(col("o_comment"))
    })
)

# Flag records for quarantine: o_orderkey is null or duplicate
w = Window.partitionBy("o_orderkey").orderBy("o_orderkey")
standardized_orders_df = standardized_orders_df.withColumn(
    "_key_count", count("*").over(w)
).withColumn(
    "_row_num", row_number().over(w)
)

# Referential integrity: o_custkey must match c_custkey in silver.customers
valid_customers_df = spark.read.table(f"{catalog}.silver.customers").select(
    col("c_custkey").alias("_valid_custkey")
)
standardized_orders_df = standardized_orders_df.join(
    valid_customers_df,
    standardized_orders_df.o_custkey == valid_customers_df._valid_custkey,
    "left"
).withColumn(
    "_invalid_custkey",
    when(
        col("o_custkey").isNotNull() & col("_valid_custkey").isNull(),
        lit("o_custkey does not match any c_custkey in silver.customers")
    ).otherwise(lit(None).cast("string"))
)

standardized_orders_df = standardized_orders_df.withColumn(
    "_quarantine_reason",
    nullif(concat_ws(
        ", ",
        when(col("o_orderkey").isNull(), lit("o_orderkey is null")).otherwise(lit(None).cast("string")),
        when((col("_key_count") > 1) & (col("_row_num") > 1), lit("duplicate o_orderkey")).otherwise(lit(None).cast("string")),
        when(col("o_totalprice") <= 0, lit("o_totalprice is zero or negative")).otherwise(lit(None).cast("string")),
        when(col("_invalid_custkey").isNotNull(), col("_invalid_custkey")).otherwise(lit(None).cast("string")),
        when(
            col("o_orderdate").isNull() |
            (col("o_orderdate") < lit("1992-01-01").cast("date")) |
            (col("o_orderdate") > lit("1998-12-31").cast("date")),
            lit("o_orderdate is not between 1992-01-01 and 1998-12-31")
        ).otherwise(lit(None).cast("string"))
    ), lit(""))
)

quarantine_orders_df = standardized_orders_df.filter(
    col("_quarantine_reason").isNotNull()
).drop("_key_count", "_valid_custkey", "_invalid_custkey", "_row_num")

clean_orders_df = standardized_orders_df.filter(
    col("o_orderkey").isNotNull()
    & (col("_row_num") == 1)
    & col("_valid_custkey").isNotNull()
    & (col("o_totalprice") > 0)
    & col("o_orderdate").isNotNull()
    & (col("o_orderdate") >= lit("1992-01-01").cast("date"))
    & (col("o_orderdate") <= lit("1998-12-31").cast("date"))
).drop("_key_count", "_quarantine_reason", "_valid_custkey", "_invalid_custkey", "_row_num")

# Write quarantined records
quarantine_orders_df.write.mode("overwrite").saveAsTable(f"{catalog}.silver.orders_quarantine")

# Write clean records
clean_orders_df.write.mode("overwrite").saveAsTable(f"{catalog}.silver.orders")

In [0]:
# silver.lineitem
from pyspark.sql.functions import *
from pyspark.sql.window import Window

lineitem_df = spark.read.table(f"{catalog}.bronze.lineitem")
standardized_lineitem_df = (
    lineitem_df
    .drop("_source_file", "_ingested_at", "_pipeline_run_id")
    .withColumns({
        "l_returnflag": trim(col("l_returnflag")),
        "l_linestatus": trim(col("l_linestatus")),
        "l_shipdate": try_to_date(col("l_shipdate"), "yyyy-MM-dd"),
        "l_commitdate": try_to_date(col("l_commitdate"), "yyyy-MM-dd"),
        "l_receiptdate": try_to_date(col("l_receiptdate"), "yyyy-MM-dd"),
        "l_shipinstruct": trim(col("l_shipinstruct")),
        "l_shipmode": trim(col("l_shipmode")),
        "l_comment": trim(col("l_comment"))
    })
)

# Flag records for quarantine: composite key (l_orderkey, l_partkey, l_suppkey) is null or duplicate
w = Window.partitionBy("l_orderkey", "l_linenumber").orderBy("l_orderkey", "l_linenumber")
standardized_lineitem_df = standardized_lineitem_df.withColumn(
    "_key_count", count("*").over(w)
).withColumn(
    "_row_num", row_number().over(w)
)

# Referential integrity: l_orderkey must match o_orderkey in silver.orders
valid_orders_df = spark.read.table(f"{catalog}.silver.orders").select(
    col("o_orderkey").alias("_valid_orderkey")
)
standardized_lineitem_df = standardized_lineitem_df.join(
    valid_orders_df,
    standardized_lineitem_df.l_orderkey == valid_orders_df._valid_orderkey,
    "left"
).withColumn(
    "_invalid_orderkey",
    when(
        col("l_orderkey").isNotNull() & col("_valid_orderkey").isNull(),
        lit("l_orderkey does not match any o_orderkey in silver.orders")
    ).otherwise(lit(None).cast("string"))
)

# Referential integrity: l_partkey must match p_partkey in silver.part
valid_parts_df = spark.read.table(f"{catalog}.silver.part").select(
    col("p_partkey").alias("_valid_partkey")
)
standardized_lineitem_df = standardized_lineitem_df.join(
    valid_parts_df,
    standardized_lineitem_df.l_partkey == valid_parts_df._valid_partkey,
    "left"
).withColumn(
    "_invalid_partkey",
    when(
        col("l_partkey").isNotNull() & col("_valid_partkey").isNull(),
        lit("l_partkey does not match any p_partkey in silver.part")
    ).otherwise(lit(None).cast("string"))
)

# Referential integrity: l_suppkey must match s_suppkey in silver.supplier
valid_suppliers_df = spark.read.table(f"{catalog}.silver.supplier").select(
    col("s_suppkey").alias("_valid_suppkey")
)
standardized_lineitem_df = standardized_lineitem_df.join(
    valid_suppliers_df,
    standardized_lineitem_df.l_suppkey == valid_suppliers_df._valid_suppkey,
    "left"
).withColumn(
    "_invalid_suppkey",
    when(
        col("l_suppkey").isNotNull() & col("_valid_suppkey").isNull(),
        lit("l_suppkey does not match any s_suppkey in silver.supplier")
    ).otherwise(lit(None).cast("string"))
)

standardized_lineitem_df = standardized_lineitem_df.withColumn(
    "_quarantine_reason",
    nullif(concat_ws(
        ", ",
        when(col("l_orderkey").isNull(), lit("l_orderkey is null")).otherwise(lit(None).cast("string")),
        when(col("l_partkey").isNull(), lit("l_partkey is null")).otherwise(lit(None).cast("string")),
        when(col("l_suppkey").isNull(), lit("l_suppkey is null")).otherwise(lit(None).cast("string")),
        when((col("_key_count") > 1) & (col("_row_num") > 1), lit("duplicate composite key (l_orderkey, l_partkey, l_suppkey)")).otherwise(lit(None).cast("string")),
        when(col("l_quantity") < 0, lit("l_quantity is negative")).otherwise(lit(None).cast("string")),
        when(col("l_extendedprice") < 0, lit("l_extendedprice is negative")).otherwise(lit(None).cast("string")),
        when((col("l_discount") < 0) | (col("l_discount") > 10), lit("l_discount is not between 0 and 10")).otherwise(lit(None).cast("string")),
        when((col("l_tax") < 0) | (col("l_tax") > 8), lit("l_tax is not between 0 and 8")).otherwise(lit(None).cast("string")),
        when(
            col("l_shipdate").isNull() |
            (col("l_shipdate") < lit("1992-01-01").cast("date")) |
            (col("l_shipdate") > lit("1998-12-31").cast("date")),
            lit("l_shipdate is not between 1992-01-01 and 1998-12-31")
        ).otherwise(lit(None).cast("string")),
        when(
            col("l_commitdate").isNull() |
            (col("l_commitdate") < lit("1992-01-01").cast("date")) |
            (col("l_commitdate") > lit("1998-12-31").cast("date")),
            lit("l_commitdate is not between 1992-01-01 and 1998-12-31")
        ).otherwise(lit(None).cast("string")),
        when(
            col("l_receiptdate").isNull() |
            (col("l_receiptdate") < lit("1992-01-01").cast("date")) |
            (col("l_receiptdate") > lit("1998-12-31").cast("date")),
            lit("l_receiptdate is not between 1992-01-01 and 1998-12-31")
        ).otherwise(lit(None).cast("string")),
        when(col("l_shipdate") > col("l_receiptdate"), lit("l_shipdate is greater than l_receiptdate")).otherwise(lit(None).cast("string")),
        when(col("_invalid_orderkey").isNotNull(), col("_invalid_orderkey")).otherwise(lit(None).cast("string")),
        when(col("_invalid_partkey").isNotNull(), col("_invalid_partkey")).otherwise(lit(None).cast("string")),
        when(col("_invalid_suppkey").isNotNull(), col("_invalid_suppkey")).otherwise(lit(None).cast("string"))
    ), lit(""))
)

quarantine_lineitem_df = standardized_lineitem_df.filter(
    col("_quarantine_reason").isNotNull()
).drop("_key_count", "_valid_orderkey", "_invalid_orderkey", "_valid_partkey", "_invalid_partkey", "_valid_suppkey", "_invalid_suppkey", "_row_num")

clean_lineitem_df = standardized_lineitem_df.filter(
    col("l_orderkey").isNotNull()
    & col("l_partkey").isNotNull()
    & col("l_suppkey").isNotNull()
    & (col("_row_num") == 1)
    & col("_valid_orderkey").isNotNull()
    & col("_valid_partkey").isNotNull()
    & col("_valid_suppkey").isNotNull()
    & (col("l_quantity") >= 0)
    & (col("l_extendedprice") >= 0)
    & (col("l_discount") >= 0) & (col("l_discount") <= 10)
    & (col("l_tax") >= 0) & (col("l_tax") <= 8)
    & col("l_shipdate").isNotNull() & (col("l_shipdate") >= lit("1992-01-01").cast("date")) & (col("l_shipdate") <= lit("1998-12-31").cast("date"))
    & col("l_commitdate").isNotNull() & (col("l_commitdate") >= lit("1992-01-01").cast("date")) & (col("l_commitdate") <= lit("1998-12-31").cast("date"))
    & col("l_receiptdate").isNotNull() & (col("l_receiptdate") >= lit("1992-01-01").cast("date")) & (col("l_receiptdate") <= lit("1998-12-31").cast("date"))
    & (col("l_shipdate") <= col("l_receiptdate"))
).drop("_key_count", "_quarantine_reason", "_valid_orderkey", "_invalid_orderkey", "_valid_partkey", "_invalid_partkey", "_valid_suppkey", "_invalid_suppkey", "_row_num")

# Write quarantined records
quarantine_lineitem_df.write.mode("overwrite").saveAsTable(f"{catalog}.silver.lineitem_quarantine")

# Write clean records
clean_lineitem_df.write.mode("overwrite").saveAsTable(f"{catalog}.silver.lineitem")

In [0]:
# Reconciliation: bronze row count == silver clean + silver quarantine, per table
from pyspark.sql.functions import *

reconciliation_tables = [
    ("customers",),
    ("lineitem",),
    ("nation",),
    ("orders",),
    ("part",),
    ("partsupp",),
    ("region",),
    ("supplier",)
]

recon_rows = []
for (tbl,) in reconciliation_tables:
    bronze_count = spark.read.table(f"{catalog}.bronze.{tbl}").count()
    silver_count = spark.read.table(f"{catalog}.silver.{tbl}").count()
    quarantine_count = spark.read.table(f"{catalog}.silver.{tbl}_quarantine").count()
    recon_rows.append((
        tbl,
        bronze_count,
        silver_count,
        quarantine_count,
        silver_count + quarantine_count,
        bronze_count == silver_count + quarantine_count
    ))

recon_df = spark.createDataFrame(
    recon_rows,
    schema="source_table string, bronze_count long, silver_count long, quarantine_count long, silver_plus_quarantine long, balanced boolean"
)

recon_df.write.mode("overwrite").saveAsTable(f"{catalog}.silver.reconciliation_summary")

In [0]:
%sql
select * from tpch_dev.silver.reconciliation_summary